# 01 — Análise exploratória da operação simulada

**Trilha CRISP-DM — compreensão dos dados.**

Este notebook lê o banco SQLite de desenvolvimento e descreve volume, status, atrasos e ocupação. Todos os dados são sintéticos. Nenhum identificador pessoal é utilizado.

Pré-requisitos: `make seed` e `pip install -r requirements-analytics.txt`.

In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RAIZ = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
BANCO = RAIZ / "data" / "svs.db"
print(BANCO, BANCO.exists())

## 1. Carga

A conexão é aberta em modo somente leitura. A análise não altera o banco.

In [ ]:
conexao = sqlite3.connect(f"file:{BANCO}?mode=ro", uri=True)

viagens = pd.read_sql_query(
    """
    SELECT v.id, v.data, v.status, v.partida_prevista, v.iniciada_em,
           v.concluida_em, ve.placa, ve.capacidade
    FROM viagem v JOIN veiculo ve ON ve.id = v.veiculo_id
    ORDER BY v.data, v.partida_prevista
    """,
    conexao,
    parse_dates=["data", "partida_prevista", "iniciada_em", "concluida_em"],
)
viagens.head()

In [ ]:
print(viagens.shape)
viagens["status"].value_counts()

## 2. Atraso e pontualidade

Atraso é `max(0, iniciada_em − partida_prevista)` em minutos. A tolerância de OTIF é uma decisão de negócio; aqui usamos 15 minutos.

In [ ]:
viagens["atraso_min"] = (
    (viagens["iniciada_em"] - viagens["partida_prevista"]).dt.total_seconds() / 60
).clip(lower=0)

iniciadas = viagens.dropna(subset=["iniciada_em"])
iniciadas["atraso_min"].describe()

In [ ]:
TOLERANCIA = 15
concluidas = viagens[(viagens["status"] == "CONCLUIDA") & viagens["iniciada_em"].notna()]

if len(concluidas):
    otif = (concluidas["atraso_min"] <= TOLERANCIA).mean() * 100
    print(f"OTIF ({len(concluidas)} concluídas): {otif:.1f}%")
else:
    print("Sem viagens concluídas no recorte.")

## 3. Ocupação

Ocupação = embarques ÷ autorizações aprovadas. O denominador é a demanda autorizada, não a capacidade do veículo.

In [ ]:
demanda = pd.read_sql_query(
    """
    SELECT a.viagem_id,
           SUM(CASE WHEN a.status = 'APROVADA' THEN 1 ELSE 0 END) AS aprovadas,
           COUNT(e.id) AS embarques
    FROM autorizacao a
    LEFT JOIN embarque e ON e.autorizacao_id = a.id
    GROUP BY a.viagem_id
    """,
    conexao,
)
demanda["ocupacao"] = (demanda["embarques"] / demanda["aprovadas"].replace(0, pd.NA) * 100).round(1)
demanda.sort_values("aprovadas", ascending=False).head(10)

In [ ]:
if len(iniciadas):
    iniciadas.sort_values("iniciada_em").plot(
        x="iniciada_em", y="atraso_min", marker="o", figsize=(9, 3),
        title="Atraso por viagem iniciada (min)", legend=False,
    )
    plt.tight_layout()
else:
    print("Sem viagens iniciadas.")

## 4. Leitura crítica

- O cenário é determinístico e pequeno: não há trânsito real, sazonalidade nem falha de campo.
- O OTIF depende da tolerância escolhida; mudar o parâmetro muda o número.
- A ocupação usa autorizações aprovadas como denominador; faltas não são registradas como evento próprio.
- Nenhuma conclusão sobre desempenho real de município pode ser extraída daqui.

Para os mesmos cálculos em formato de linha de comando, use `analytics/src/indicadores.py`.